# Cross-Dataset Evaluation: UTD-MHAD → CZU-MHAD

Test the UTD-MHAD deployment models (trained on **all** UTD-MHAD data, no video)
on **unseen CZU-MHAD data** using only the overlapping activity classes.

**2-Class Setup (merged circle variants):**

| Merged Label | UTD-MHAD sources | CZU-MHAD sources |
|---|---|---|
| **Clap** (0) | clap | clap |
| **Draw Circle** (1) | draw_circle_CW + draw_circle_CCW | draw_circle_right + draw_circle_left |

Both CW/CCW (UTD) and right/left hand (CZU) circle variants are merged into a single
"draw circle" class. This avoids the ambiguous CW↔right / CCW↔left mapping problem.

**Pipeline:** Load CZU-MHAD features → filter to matching classes → merge circle variants →
apply independent scaler → apply each metaheuristic's mask → classify with deployment model →
merge model's CW+CCW predictions into "circle" → report accuracy & timing.

In [118]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [119]:
import numpy as np
import pandas as pd
import joblib
import time
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cuda
GPU: NVIDIA GeForce RTX 4090


## Configuration & Paths

In [120]:
# UTD-MHAD deployment artifacts (no-video model trained on all subjects)
UTD_RESULTS_ROOT = Path("results_all_no_video")

# CZU-MHAD features
CZU_FEATURE_DIR = Path("../czu-mhad/features_new")

# Output directory for cross-dataset results
CROSS_RESULTS = Path("results_cross_dataset")
CROSS_RESULTS.mkdir(exist_ok=True)

# All 14 metaheuristic method names
META_METHODS = [
    'meta_BAT', 'meta_CS', 'meta_DE', 'meta_FFA', 'meta_GA', 'meta_GWO',
    'meta_HHO', 'meta_JAYA', 'meta_MFO', 'meta_MVO', 'meta_PSO',
    'meta_SCA', 'meta_SSA', 'meta_WOA'
]

USE_CUDA_SYNC = torch.cuda.is_available()

def sync_cuda():
    if USE_CUDA_SYNC:
        torch.cuda.synchronize()

print(f"UTD-MHAD results: {UTD_RESULTS_ROOT}")
print(f"CZU-MHAD features: {CZU_FEATURE_DIR}")
print(f"Cross-dataset output: {CROSS_RESULTS}")

UTD-MHAD results: results_all_no_video
CZU-MHAD features: ../czu-mhad/features_new
Cross-dataset output: results_cross_dataset


## Model Definition (must match UTD-MHAD training)

In [121]:
class SimpleNN(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        hidden1 = max(128, min(512, input_dim * 2))
        hidden2 = max(64, min(256, hidden1 // 2))
        self.classifier = nn.Sequential(
            nn.Linear(input_dim, hidden1),
            nn.BatchNorm1d(hidden1),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(hidden1, hidden2),
            nn.BatchNorm1d(hidden2),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden2, num_classes)
        )
    def forward(self, x):
        return self.classifier(x)

print("Model architecture defined (matches UTD-MHAD deployment)")

Model architecture defined (matches UTD-MHAD deployment)


## Load UTD-MHAD Artifacts

In [122]:
# Load the scaler fitted on UTD-MHAD training data
utd_scaler = joblib.load(UTD_RESULTS_ROOT / "scaler.pkl")
print("Loaded UTD-MHAD scaler")

# Load UTD-MHAD label encoder to get class names
utd_le = joblib.load(Path("features_new") / "label_encoder.pkl")
utd_classes = list(utd_le.classes_)
print(f"UTD-MHAD classes ({len(utd_classes)}): {utd_classes}")

# Identify UTD-MHAD class indices for matching classes
utd_clap_idx = utd_classes.index(4)                  # clap
utd_draw_cw_idx = utd_classes.index(9)               # draw circle CW
utd_draw_ccw_idx = utd_classes.index(10)             # draw circle CCW

print(f"\nUTD-MHAD matching class indices:")
print(f"  Clap:             {utd_clap_idx} ('{utd_classes[utd_clap_idx]}')")
print(f"  Draw circle CW:   {utd_draw_cw_idx} ('{utd_classes[utd_draw_cw_idx]}')")
print(f"  Draw circle CCW:  {utd_draw_ccw_idx} ('{utd_classes[utd_draw_ccw_idx]}')")
print(f"  → Both CW and CCW will be merged into 'draw_circle' (label 1)")

# How many total classes did UTD-MHAD model output?
UTD_NUM_CLASSES = len(utd_classes)
print(f"  Total UTD output classes: {UTD_NUM_CLASSES}")

# Define the 2-class merged labels
LABEL_CLAP = 0
LABEL_CIRCLE = 1
MERGED_CLASS_NAMES = ['clap', 'draw_circle']
print(f"\nMerged 2-class labels: {MERGED_CLASS_NAMES}")

Loaded UTD-MHAD scaler
UTD-MHAD classes (27): [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24), np.int64(25), np.int64(26), np.int64(27)]

UTD-MHAD matching class indices:
  Clap:             3 ('4')
  Draw circle CW:   8 ('9')
  Draw circle CCW:  9 ('10')
  → Both CW and CCW will be merged into 'draw_circle' (label 1)
  Total UTD output classes: 27

Merged 2-class labels: ['clap', 'draw_circle']


## Load CZU-MHAD Data

In [123]:
# Load CZU-MHAD features
czu_X_feat = joblib.load(CZU_FEATURE_DIR / "X_feat.pkl")
czu_y = np.load(CZU_FEATURE_DIR / "y.npy")
czu_subjects = np.load(CZU_FEATURE_DIR / "subjects.npy")
czu_le = joblib.load(CZU_FEATURE_DIR / "label_encoder.pkl")

czu_classes = list(czu_le.classes_)
print(f"CZU-MHAD: {len(czu_X_feat)} samples, {len(czu_classes)} classes, "
      f"{len(np.unique(czu_subjects))} subjects")
print(f"CZU-MHAD classes: {czu_classes}")

# Verify feature dimensions match
czu_first = czu_X_feat[0]
czu_dims = {
    'depth': czu_first['depth_feat'].shape[0],
    'sensor': czu_first['sensor_feat'].shape[0],
    'skeleton': czu_first['skeleton_feat'].shape[0]
}
print(f"\nCZU-MHAD feature dims: {czu_dims}")
print(f"CZU-MHAD total features: {sum(czu_dims.values())}")

# Load UTD feature dims for comparison
utd_first = joblib.load(Path("features_new") / "X_feat.pkl")[0]
utd_dims = {
    'depth': utd_first['depth_feat'].shape[0],
    'sensor': utd_first['sensor_feat'].shape[0],
    'skeleton': utd_first['skeleton_feat'].shape[0]
}
UTD_TOTAL_FEATURES = sum(utd_dims.values())
print(f"UTD-MHAD feature dims: {utd_dims}")
print(f"UTD-MHAD total features: {UTD_TOTAL_FEATURES}")

# Check compatibility
assert czu_dims == utd_dims, (
    f"Feature dimension mismatch! CZU: {czu_dims} vs UTD: {utd_dims}. "
    f"Cannot proceed — feature extractors produced different-sized vectors."
)
print("\nFeature dimensions MATCH — cross-dataset evaluation is possible!")

CZU-MHAD: 1165 samples, 22 classes, 5 subjects
CZU-MHAD classes: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22)]

CZU-MHAD feature dims: {'depth': 216, 'sensor': 652, 'skeleton': 1645}
CZU-MHAD total features: 2513
UTD-MHAD feature dims: {'depth': 216, 'sensor': 652, 'skeleton': 1645}
UTD-MHAD total features: 2513

Feature dimensions MATCH — cross-dataset evaluation is possible!


## Identify Matching Classes & Build Mappings

In [124]:
# CZU-MHAD class names — find the matching ones
print("CZU-MHAD label encoder classes:")
for i, name in enumerate(czu_classes):
    print(f"  {i}: {name}")

# Auto-detect CZU class indices by keyword matching
czu_clap_idx = 14
czu_circle_right_idx = 8
czu_circle_left_idx = 9

print(f"\nDetected CZU-MHAD matching classes:")
print(f"  Clap:                {czu_clap_idx} ('{czu_classes[czu_clap_idx] if czu_clap_idx is not None else 'NOT FOUND'}')")
print(f"  Draw circle right:   {czu_circle_right_idx} ('{czu_classes[czu_circle_right_idx] if czu_circle_right_idx is not None else 'NOT FOUND'}')")
print(f"  Draw circle left:    {czu_circle_left_idx} ('{czu_classes[czu_circle_left_idx] if czu_circle_left_idx is not None else 'NOT FOUND'}')")

assert czu_clap_idx is not None, "Could not find 'clap' in CZU-MHAD classes!"
assert czu_circle_right_idx is not None, "Could not find 'circle right' in CZU-MHAD classes!"
assert czu_circle_left_idx is not None, "Could not find 'circle left' in CZU-MHAD classes!"

# =====================================================================
# Single mapping: merge both circle variants into one class
# =====================================================================
# CZU label → merged 2-class label
czu_to_merged = {
    czu_clap_idx: LABEL_CLAP,           # clap → 0
    czu_circle_right_idx: LABEL_CIRCLE,  # circle right → 1
    czu_circle_left_idx: LABEL_CIRCLE,   # circle left  → 1
}

# UTD model output → merged 2-class label
utd_to_merged = {
    utd_clap_idx: LABEL_CLAP,       # clap → 0
    utd_draw_cw_idx: LABEL_CIRCLE,   # draw circle CW  → 1
    utd_draw_ccw_idx: LABEL_CIRCLE,  # draw circle CCW → 1
}

# All other UTD classes map to -1 (wrong / outside matching classes)

print(f"\nMerged label mapping:")
print(f"  CZU clap ({czu_clap_idx}) → {LABEL_CLAP} (clap)")
print(f"  CZU circle right ({czu_circle_right_idx}) → {LABEL_CIRCLE} (draw_circle)")
print(f"  CZU circle left ({czu_circle_left_idx}) → {LABEL_CIRCLE} (draw_circle)")
print(f"  UTD clap ({utd_clap_idx}) → {LABEL_CLAP} (clap)")
print(f"  UTD CW ({utd_draw_cw_idx}) → {LABEL_CIRCLE} (draw_circle)")
print(f"  UTD CCW ({utd_draw_ccw_idx}) → {LABEL_CIRCLE} (draw_circle)")
print(f"  UTD all other classes → -1 (outside)")

CZU-MHAD label encoder classes:
  0: 1
  1: 2
  2: 3
  3: 4
  4: 5
  5: 6
  6: 7
  7: 8
  8: 9
  9: 10
  10: 11
  11: 12
  12: 13
  13: 14
  14: 15
  15: 16
  16: 17
  17: 18
  18: 19
  19: 20
  20: 21
  21: 22

Detected CZU-MHAD matching classes:
  Clap:                14 ('15')
  Draw circle right:   8 ('9')
  Draw circle left:    9 ('10')

Merged label mapping:
  CZU clap (14) → 0 (clap)
  CZU circle right (8) → 1 (draw_circle)
  CZU circle left (9) → 1 (draw_circle)
  UTD clap (3) → 0 (clap)
  UTD CW (8) → 1 (draw_circle)
  UTD CCW (9) → 1 (draw_circle)
  UTD all other classes → -1 (outside)


## Prepare CZU-MHAD Test Data

In [125]:
# Filter CZU-MHAD to only matching classes
matching_czu_indices = [czu_clap_idx, czu_circle_right_idx, czu_circle_left_idx]
keep_mask = np.isin(czu_y, matching_czu_indices)

czu_X_feat_matched = [czu_X_feat[i] for i in range(len(czu_X_feat)) if keep_mask[i]]
czu_y_matched = czu_y[keep_mask]

print(f"CZU-MHAD samples with matching classes: {len(czu_y_matched)} / {len(czu_y)}")
for idx in matching_czu_indices:
    count = np.sum(czu_y_matched == idx)
    print(f"  '{czu_classes[idx]}': {count} samples")

# Build unified feature vectors (depth + sensor + skeleton, same order as UTD)
czu_X_unified = []
for sample in czu_X_feat_matched:
    feat_vector = np.concatenate([
        sample['depth_feat'],
        sample['sensor_feat'],
        sample['skeleton_feat']
    ])
    czu_X_unified.append(feat_vector)
czu_X_unified = np.array(czu_X_unified)

print(f"\nCZU-MHAD unified feature shape: {czu_X_unified.shape}")
print(f"Expected features: {UTD_TOTAL_FEATURES}")
assert czu_X_unified.shape[1] == UTD_TOTAL_FEATURES, "Feature dimension mismatch!"

# Independent scaling
czu_scaler = StandardScaler()
czu_X_scaled = czu_scaler.fit_transform(czu_X_unified)
print("Applied INDEPENDENT StandardScaler to CZU-MHAD features (fit on CZU-MHAD only)")
print(f"  CZU-MHAD scaled — mean: {czu_X_scaled.mean():.4f}, std: {czu_X_scaled.std():.4f}")

# Create merged ground truth labels (clap=0, draw_circle=1)
czu_y_merged = np.array([czu_to_merged[label] for label in czu_y_matched])

print(f"\nMerged 2-class label distribution:")
print(f"  clap (0):        {np.sum(czu_y_merged == LABEL_CLAP)} samples")
print(f"  draw_circle (1): {np.sum(czu_y_merged == LABEL_CIRCLE)} samples")
print(f"  Total:           {len(czu_y_merged)} samples")

CZU-MHAD samples with matching classes: 158 / 1165
  '15': 53 samples
  '9': 53 samples
  '10': 52 samples

CZU-MHAD unified feature shape: (158, 2513)
Expected features: 2513
Applied INDEPENDENT StandardScaler to CZU-MHAD features (fit on CZU-MHAD only)
  CZU-MHAD scaled — mean: 0.0000, std: 0.9139

Merged 2-class label distribution:
  clap (0):        53 samples
  draw_circle (1): 105 samples
  Total:           158 samples


## Evaluate All 14 Metaheuristics + Baseline on CZU-MHAD

In [126]:
def load_and_evaluate(method_name, X_test_scaled, y_test_merged, total_features, num_classes):
    """
    Load a deployment mask + model from UTD-MHAD results,
    classify CZU-MHAD test data with output logits restricted to
    only the 3 matching UTD classes (clap, CW, CCW), then merge
    CW/CCW into a single 'draw_circle' class.
    """
    method_dir = UTD_RESULTS_ROOT / method_name
    
    # Load feature mask
    mask = np.load(method_dir / "deployment_mask.npy")
    n_selected = int(np.sum(mask))
    
    # Load model
    model = SimpleNN(n_selected, num_classes).to(DEVICE)
    model.load_state_dict(torch.load(method_dir / "deployment_model.pth", map_location=DEVICE))
    model.eval()
    
    # Build logit mask: only allow clap, CW, CCW outputs
    allowed_classes = [utd_clap_idx, utd_draw_cw_idx, utd_draw_ccw_idx]
    logit_mask = torch.full((num_classes,), float('-inf'), device=DEVICE)
    for c in allowed_classes:
        logit_mask[c] = 0.0  # keep these logits unchanged
    
    # GPU warmup
    dummy = torch.FloatTensor(X_test_scaled[0:1, mask]).to(DEVICE)
    with torch.no_grad():
        for _ in range(50):
            out = model(dummy)
            _ = torch.argmax(out + logit_mask, dim=1)
    sync_cuda()
    
    # Classify all samples
    raw_preds = []
    mask_times = []
    class_times = []
    
    for i in range(len(X_test_scaled)):
        sample = X_test_scaled[i:i+1]
        
        # Feature mask application
        start_mask = time.perf_counter()
        sample_masked = sample[:, mask]
        end_mask = time.perf_counter()
        
        # Classification (with logit masking)
        sample_tensor = torch.FloatTensor(sample_masked).to(DEVICE)
        sync_cuda()
        
        start_class = time.perf_counter()
        with torch.no_grad():
            output = model(sample_tensor)
            masked_output = output + logit_mask  # -inf kills non-matching classes
            pred = torch.argmax(masked_output, dim=1).cpu().item()
        sync_cuda()
        end_class = time.perf_counter()
        
        raw_preds.append(pred)
        mask_times.append((end_mask - start_mask) * 1000)
        class_times.append((end_class - start_class) * 1000)
    
    raw_preds = np.array(raw_preds)
    total_times = [m + c for m, c in zip(mask_times, class_times)]
    
    # Merge CW/CCW into single 'draw_circle' label
    # With logit masking, all predictions are guaranteed to be clap/CW/CCW
    merged_preds = np.full_like(raw_preds, -1)
    merged_preds[raw_preds == utd_clap_idx] = LABEL_CLAP
    merged_preds[raw_preds == utd_draw_cw_idx] = LABEL_CIRCLE
    merged_preds[raw_preds == utd_draw_ccw_idx] = LABEL_CIRCLE
    
    return {
        'raw_preds': raw_preds,
        'merged_preds': merged_preds,
        'n_features': n_selected,
        'feature_retention_pct': n_selected / total_features * 100,
        'avg_mask_time_ms': np.mean(mask_times),
        'avg_class_time_ms': np.mean(class_times),
        'avg_total_time_ms': np.mean(total_times),
        'std_total_time_ms': np.std(total_times),
    }

print("Evaluation function defined (logit-masked to 3 UTD classes, merged to 2)")

Evaluation function defined (logit-masked to 3 UTD classes, merged to 2)


In [127]:
# Run evaluation for baseline + all 14 metaheuristics
all_methods = ['baseline'] + META_METHODS
results = {}

print("=" * 90)
print("CROSS-DATASET EVALUATION: UTD-MHAD models → CZU-MHAD test data (2-class)")
print("=" * 90)

for method in all_methods:
    print(f"\n  Evaluating {method}...")
    
    res = load_and_evaluate(method, czu_X_scaled, czu_y_merged,
                            UTD_TOTAL_FEATURES, UTD_NUM_CLASSES)
    
    # Accuracy on merged 2-class labels
    acc = accuracy_score(czu_y_merged, res['merged_preds'])
    f1 = f1_score(czu_y_merged, res['merged_preds'], average='macro', zero_division=0)
    res['accuracy'] = acc
    res['f1_macro'] = f1
    results[method] = res
    
    # Count predictions outside matching classes (pred == -1)
    outside = np.sum(res['merged_preds'] == -1)
    
    print(f"    Acc={acc*100:.2f}%, F1={f1*100:.2f}%, "
          f"Features: {res['n_features']}/{UTD_TOTAL_FEATURES} "
          f"({res['feature_retention_pct']:.1f}%), "
          f"Avg inference: {res['avg_total_time_ms']:.4f} ms, "
          f"Outside preds: {outside}/{len(czu_y_merged)}")

print("\n" + "=" * 90)
print("ALL EVALUATIONS COMPLETE")
print("=" * 90)

CROSS-DATASET EVALUATION: UTD-MHAD models → CZU-MHAD test data (2-class)

  Evaluating baseline...
    Acc=55.06%, F1=53.91%, Features: 2513/2513 (100.0%), Avg inference: 0.1665 ms, Outside preds: 0/158

  Evaluating meta_BAT...
    Acc=57.59%, F1=49.58%, Features: 1203/2513 (47.9%), Avg inference: 0.1760 ms, Outside preds: 0/158

  Evaluating meta_CS...
    Acc=52.53%, F1=45.43%, Features: 1226/2513 (48.8%), Avg inference: 0.1774 ms, Outside preds: 0/158

  Evaluating meta_DE...
    Acc=58.23%, F1=55.37%, Features: 1212/2513 (48.2%), Avg inference: 0.1978 ms, Outside preds: 0/158

  Evaluating meta_FFA...
    Acc=53.80%, F1=45.70%, Features: 1287/2513 (51.2%), Avg inference: 0.1695 ms, Outside preds: 0/158

  Evaluating meta_GA...
    Acc=55.70%, F1=55.11%, Features: 1224/2513 (48.7%), Avg inference: 0.1651 ms, Outside preds: 0/158

  Evaluating meta_GWO...
    Acc=47.47%, F1=40.23%, Features: 698/2513 (27.8%), Avg inference: 0.2569 ms, Outside preds: 0/158

  Evaluating meta_HHO...
 

## Results (2-Class: Clap vs Draw Circle)

In [128]:
# Results are already in `results` dict — no mapping selection needed
# Just identify the best metaheuristic
best_meta = max(META_METHODS, key=lambda m: results[m]['accuracy'])
best_meta_res = results[best_meta]
baseline_res = results['baseline']

print(f"Best metaheuristic: {best_meta} (Acc={best_meta_res['accuracy']*100:.2f}%)")
print(f"Baseline: Acc={baseline_res['accuracy']*100:.2f}%")

Best metaheuristic: meta_SCA (Acc=65.19%)
Baseline: Acc=55.06%


## Results: All Methods with Best Mapping

In [129]:
# Build comprehensive results table
rows = []
for method in all_methods:
    r = results[method]
    outside = np.sum(r['merged_preds'] == -1)
    rows.append({
        'Method': method,
        'Accuracy (%)': r['accuracy'] * 100,
        'F1 Macro (%)': r['f1_macro'] * 100,
        'Features': r['n_features'],
        'Feature Retention (%)': r['feature_retention_pct'],
        'Mask Time (ms)': r['avg_mask_time_ms'],
        'Class Time (ms)': r['avg_class_time_ms'],
        'Total Time (ms)': r['avg_total_time_ms'],
        'Time Std (ms)': r['std_total_time_ms'],
        'Outside Preds': outside,
        'Outside Preds (%)': outside / len(czu_y_merged) * 100,
    })

df_results = pd.DataFrame(rows).round(4)
df_results = df_results.sort_values('Accuracy (%)', ascending=False)

print("\n" + "=" * 100)
print("CROSS-DATASET RESULTS (2-Class: Clap vs Draw Circle)")
print("=" * 100)
print(df_results.to_string(index=False))

df_results.to_csv(CROSS_RESULTS / "cross_dataset_results_2class.csv", index=False)
print(f"\nSaved: {CROSS_RESULTS / 'cross_dataset_results_2class.csv'}")


CROSS-DATASET RESULTS (2-Class: Clap vs Draw Circle)
   Method  Accuracy (%)  F1 Macro (%)  Features  Feature Retention (%)  Mask Time (ms)  Class Time (ms)  Total Time (ms)  Time Std (ms)  Outside Preds  Outside Preds (%)
 meta_SCA       65.1899       61.1412       280                11.1421          0.0075           0.1580           0.1655         0.1273              0                0.0
 meta_MFO       60.1266       54.1607      1236                49.1842          0.0082           0.1462           0.1543         0.0343              0                0.0
 meta_SSA       60.1266       52.5887      1197                47.6323          0.0087           0.1666           0.1754         0.1530              0                0.0
  meta_DE       58.2278       55.3672      1212                48.2292          0.0087           0.1891           0.1978         0.2002              0                0.0
 meta_BAT       57.5949       49.5785      1203                47.8711          0.0087          

## Best Metaheuristic — Detailed Analysis

In [130]:
# Best metaheuristic detailed analysis
print("=" * 85)
print("BEST METAHEURISTIC FOR CROSS-DATASET TRANSFER (2-Class)")
print("=" * 85)

print(f"\nBest metaheuristic: {best_meta}")
print(f"  Accuracy:          {best_meta_res['accuracy']*100:.2f}%")
print(f"  F1 Macro:          {best_meta_res['f1_macro']*100:.2f}%")
print(f"  Features:          {best_meta_res['n_features']}/{UTD_TOTAL_FEATURES} "
      f"({best_meta_res['feature_retention_pct']:.1f}%)")
print(f"  Avg mask time:     {best_meta_res['avg_mask_time_ms']:.4f} ms")
print(f"  Avg class time:    {best_meta_res['avg_class_time_ms']:.4f} ms")
print(f"  Avg total time:    {best_meta_res['avg_total_time_ms']:.4f} ms")

print(f"\nBaseline (all features):")
print(f"  Accuracy:          {baseline_res['accuracy']*100:.2f}%")
print(f"  F1 Macro:          {baseline_res['f1_macro']*100:.2f}%")
print(f"  Features:          {baseline_res['n_features']}/{UTD_TOTAL_FEATURES} (100%)")
print(f"  Avg class time:    {baseline_res['avg_class_time_ms']:.4f} ms")

speedup = baseline_res['avg_total_time_ms'] / best_meta_res['avg_total_time_ms'] if best_meta_res['avg_total_time_ms'] > 0 else float('inf')
acc_diff = best_meta_res['accuracy'] - baseline_res['accuracy']

print(f"\nComparison:")
print(f"  Accuracy difference:  {acc_diff*100:+.2f}% ({'better' if acc_diff > 0 else 'worse'})")
print(f"  Inference speedup:    {speedup:.2f}x")
print(f"  Feature reduction:    {100 - best_meta_res['feature_retention_pct']:.1f}%")

BEST METAHEURISTIC FOR CROSS-DATASET TRANSFER (2-Class)

Best metaheuristic: meta_SCA
  Accuracy:          65.19%
  F1 Macro:          61.14%
  Features:          280/2513 (11.1%)
  Avg mask time:     0.0075 ms
  Avg class time:    0.1580 ms
  Avg total time:    0.1655 ms

Baseline (all features):
  Accuracy:          55.06%
  F1 Macro:          53.91%
  Features:          2513/2513 (100%)
  Avg class time:    0.1561 ms

Comparison:
  Accuracy difference:  +10.13% (better)
  Inference speedup:    1.01x
  Feature reduction:    88.9%


In [131]:
# Per-class classification report for best metaheuristic
best_preds = results[best_meta]['merged_preds']

print(f"\nPer-class report ({best_meta}, 2-class merged):")
print("-" * 60)
print(classification_report(
    czu_y_merged, best_preds,
    labels=[LABEL_CLAP, LABEL_CIRCLE],
    target_names=MERGED_CLASS_NAMES,
    zero_division=0
))

# Confusion matrix
cm = confusion_matrix(czu_y_merged, best_preds, labels=[LABEL_CLAP, LABEL_CIRCLE])
print("Confusion matrix (rows=true, cols=pred):")
cm_df = pd.DataFrame(cm, index=MERGED_CLASS_NAMES, columns=MERGED_CLASS_NAMES)
print(cm_df)

# Predictions outside matching classes
outside = np.sum(best_preds == -1)
print(f"\nPredictions outside matching classes (model predicted a non-clap/non-circle UTD class):")
print(f"  {outside}/{len(best_preds)} ({outside/len(best_preds)*100:.1f}%)")

# Show which UTD classes the model predicted
raw = results[best_meta]['raw_preds']
print(f"\nRaw UTD prediction distribution:")
for cls_idx in sorted(np.unique(raw)):
    count = np.sum(raw == cls_idx)
    name = utd_classes[cls_idx] if cls_idx < len(utd_classes) else f"unknown({cls_idx})"
    is_match = "✓" if cls_idx in [utd_clap_idx, utd_draw_cw_idx, utd_draw_ccw_idx] else "✗"
    print(f"  {is_match} UTD class {cls_idx:2d} ({name}): {count:4d} samples")


Per-class report (meta_SCA, 2-class merged):
------------------------------------------------------------
              precision    recall  f1-score   support

        clap       0.48      0.49      0.49        53
 draw_circle       0.74      0.73      0.74       105

    accuracy                           0.65       158
   macro avg       0.61      0.61      0.61       158
weighted avg       0.65      0.65      0.65       158

Confusion matrix (rows=true, cols=pred):
             clap  draw_circle
clap           26           27
draw_circle    28           77

Predictions outside matching classes (model predicted a non-clap/non-circle UTD class):
  0/158 (0.0%)

Raw UTD prediction distribution:
  ✓ UTD class  3 (4):   54 samples
  ✓ UTD class  8 (9):   39 samples
  ✓ UTD class  9 (10):   65 samples


In [132]:
# Same report for baseline
baseline_preds = results['baseline']['merged_preds']

print(f"\nPer-class report (baseline, 2-class merged):")
print("-" * 60)
print(classification_report(
    czu_y_merged, baseline_preds,
    labels=[LABEL_CLAP, LABEL_CIRCLE],
    target_names=MERGED_CLASS_NAMES,
    zero_division=0
))

cm_base = confusion_matrix(czu_y_merged, baseline_preds, labels=[LABEL_CLAP, LABEL_CIRCLE])
print("Confusion matrix (rows=true, cols=pred):")
cm_base_df = pd.DataFrame(cm_base, index=MERGED_CLASS_NAMES, columns=MERGED_CLASS_NAMES)
print(cm_base_df)

outside_base = np.sum(baseline_preds == -1)
print(f"\nPredictions outside matching classes: {outside_base}/{len(baseline_preds)} "
      f"({outside_base/len(baseline_preds)*100:.1f}%)")

# Raw prediction distribution
raw_base = results['baseline']['raw_preds']
print(f"\nRaw UTD prediction distribution (baseline):")
for cls_idx in sorted(np.unique(raw_base)):
    count = np.sum(raw_base == cls_idx)
    name = utd_classes[cls_idx] if cls_idx < len(utd_classes) else f"unknown({cls_idx})"
    is_match = "✓" if cls_idx in [utd_clap_idx, utd_draw_cw_idx, utd_draw_ccw_idx] else "✗"
    print(f"  {is_match} UTD class {cls_idx:2d} ({name}): {count:4d} samples")


Per-class report (baseline, 2-class merged):
------------------------------------------------------------
              precision    recall  f1-score   support

        clap       0.39      0.58      0.47        53
 draw_circle       0.72      0.53      0.61       105

    accuracy                           0.55       158
   macro avg       0.55      0.56      0.54       158
weighted avg       0.61      0.55      0.56       158

Confusion matrix (rows=true, cols=pred):
             clap  draw_circle
clap           31           22
draw_circle    49           56

Predictions outside matching classes: 0/158 (0.0%)

Raw UTD prediction distribution (baseline):
  ✓ UTD class  3 (4):   80 samples
  ✓ UTD class  8 (9):   17 samples
  ✓ UTD class  9 (10):   61 samples


## All Methods Comparison

In [133]:
# Bar chart: all methods accuracy
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(16, 7))
methods_sorted = df_results.sort_values('Accuracy (%)', ascending=False)['Method'].tolist()
accs = df_results.set_index('Method').loc[methods_sorted, 'Accuracy (%)']

bars = ax.bar(range(len(methods_sorted)), accs.values, alpha=0.7, color='steelblue')
ax.set_xticks(range(len(methods_sorted)))
ax.set_xticklabels(methods_sorted, rotation=60, ha='right')
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('Cross-Dataset Accuracy: UTD-MHAD → CZU-MHAD (2-Class)', fontsize=14)
ax.axhline(y=50, color='red', linestyle='--', alpha=0.5, label='Random baseline (50%)')
ax.legend()
ax.grid(axis='y', alpha=0.3)

for bar in bars:
    val = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., val + 0.5,
            f'{val:.1f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

plt.tight_layout()
plt.savefig(CROSS_RESULTS / 'cross_dataset_accuracy_2class.png', dpi=300, bbox_inches='tight')
plt.close()
print(f"Saved: {CROSS_RESULTS / 'cross_dataset_accuracy_2class.png'}")

# Save final results
df_results.to_csv(CROSS_RESULTS / "cross_dataset_results_2class.csv", index=False)
print(f"Saved: {CROSS_RESULTS / 'cross_dataset_results_2class.csv'}")

Saved: results_cross_dataset/cross_dataset_accuracy_2class.png
Saved: results_cross_dataset/cross_dataset_results_2class.csv


## Timing Summary

In [134]:
print("\n" + "=" * 85)
print("INFERENCE TIMING SUMMARY (CZU-MHAD cross-dataset, 2-class)")
print("=" * 85)

print(f"""
┌───────────────────────────────────┬──────────────┬──────────────────────┐
│ Component                         │ Baseline     │ {best_meta:<20s} │
├───────────────────────────────────┼──────────────┼──────────────────────┤
│ Mask Application                  │     ---      │ {best_meta_res['avg_mask_time_ms']:>10.4f} ms        │
│ NN Forward Pass                   │ {baseline_res['avg_class_time_ms']:>10.4f} ms│ {best_meta_res['avg_class_time_ms']:>10.4f} ms        │
│ Total per-sample                  │ {baseline_res['avg_total_time_ms']:>10.4f} ms│ {best_meta_res['avg_total_time_ms']:>10.4f} ms        │
├───────────────────────────────────┼──────────────┼──────────────────────┤
│ Inference Speedup                 │    1.00x     │ {speedup:>10.2f}x          │
├───────────────────────────────────┼──────────────┼──────────────────────┤
│ CZU-MHAD Accuracy (2-class)       │ {baseline_res['accuracy']*100:>9.2f}%   │ {best_meta_res['accuracy']*100:>9.2f}%           │
│ Features Used                     │ {UTD_TOTAL_FEATURES:>10d}   │ {best_meta_res['n_features']:>10d}           │
│ Feature Retention                 │     100.0%   │ {best_meta_res['feature_retention_pct']:>10.1f}%          │
│ Test Samples                      │ {len(czu_y_merged):>10d}   │ {len(czu_y_merged):>10d}           │
└───────────────────────────────────┴──────────────┴──────────────────────┘
""")

print("Cross-dataset evaluation complete!")
print(f"All results saved to: {CROSS_RESULTS}/")


INFERENCE TIMING SUMMARY (CZU-MHAD cross-dataset, 2-class)

┌───────────────────────────────────┬──────────────┬──────────────────────┐
│ Component                         │ Baseline     │ meta_SCA             │
├───────────────────────────────────┼──────────────┼──────────────────────┤
│ Mask Application                  │     ---      │     0.0075 ms        │
│ NN Forward Pass                   │     0.1561 ms│     0.1580 ms        │
│ Total per-sample                  │     0.1665 ms│     0.1655 ms        │
├───────────────────────────────────┼──────────────┼──────────────────────┤
│ Inference Speedup                 │    1.00x     │       1.01x          │
├───────────────────────────────────┼──────────────┼──────────────────────┤
│ CZU-MHAD Accuracy (2-class)       │     55.06%   │     65.19%           │
│ Features Used                     │       2513   │        280           │
│ Feature Retention                 │     100.0%   │       11.1%          │
│ Test Samples             